Bring in functions from lab

In [32]:
import numpy as np
rng = np.random.default_rng(100)
def purchase_probability(price):
    price = np.asarray(price, dtype=float)
    return (
        0.55 * np.exp(-np.logaddexp(0.0, (price - 13.9) / 0.8))
        + 0.30 * np.exp(-np.logaddexp(0.0, (price - 30.9) / 1.2))
        + 0.15 * np.exp(-np.logaddexp(0.0, (price - 58.4) / 1.8))
    )

def sales(price=14.0, rng=None, mean_customers=30):
    price = float(price)
    if not np.isfinite(price):
        raise ValueError("price must be finite")
    if price < 0:
        raise ValueError("price must be nonnegative")
    mean_customers = float(mean_customers)
    if not np.isfinite(mean_customers) or mean_customers < 0:
        raise ValueError("mean_customers must be finite and nonnegative")
    rng = np.random.default_rng(rng)
    customer_count = rng.poisson(mean_customers)
    probability = purchase_probability(price)
    return rng.binomial(1, probability, size=customer_count).astype(np.int8)

we know its between 0 and 70 so lets try spacings of 5 between those as our coarse grid

In [33]:
prices = np.arange(0, 71, 5)
customers_per_price = 200
prices

array([ 0,  5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70])

run revenue at each price point

In [34]:
revenue_by_price = {}
total_customers = 0
total_revenue = 0
for price in prices:
    purchases = sales(price, mean_customers=customers_per_price, rng=rng)
    customer_revenue = purchases * price
    revenue_by_price[price] = customer_revenue
    total_customers += len(purchases)
    total_revenue += customer_revenue.sum()

print("Total customers:", total_customers)
print("Total revenue:", total_revenue)

Total customers: 2970
Total revenue: 17255


Relevant Statistics at each price point

In [35]:
import pandas as pd

rows = []
for price in prices:
    customer_revenue = revenue_by_price[price]
    mean_revenue = customer_revenue.mean()
    variance = customer_revenue.var(ddof=1)
    customers = len(customer_revenue)
    rows.append([price, mean_revenue, variance, customers])

columns = ["price", "mean_revenue", "variance", "customers"]
price_summary = pd.DataFrame(rows, columns=columns)
price_summary.round(2)

,price,mean_revenue,variance,customers
0,0,0.00,0.00,216
1,5,5.00,0.00,176
2,10,9.95,0.46,216
3,15,8.28,55.93,192
4,20,9.05,99.57,210
5,25,12.68,156.98,205
6,30,7.54,170.20,195
7,35,6.05,176.00,191
8,40,5.03,176.87,175
9,45,7.01,267.66,199


In [36]:
leader_index = price_summary["mean_revenue"].idxmax()
leader_price = price_summary.loc[leader_index, "price"]
print("Provisional leader:", leader_price)

Provisional leader: 25


Now use 95% confidence intervals as we are early right now and dont need too high precision which will be later of the leader vs every other price point

In [37]:
leader_index = price_summary["mean_revenue"].idxmax()
leader_price = price_summary.loc[leader_index, "price"]
leader_revenue = revenue_by_price[leader_price]
leader_mean = leader_revenue.mean()
leader_variance = leader_revenue.var(ddof=1)
leader_customers = len(leader_revenue)
comparison_rows = []
promising_prices = []

for price in prices:
    customer_revenue = revenue_by_price[price]
    difference = leader_mean - customer_revenue.mean()
    if price == leader_price:
        ci_low = np.nan
        ci_high = np.nan
        status = "Leader"
    else:
        variance = customer_revenue.var(ddof=1)
        customers = len(customer_revenue)
        variance_difference = leader_variance / leader_customers + variance / customers
        standard_error = np.sqrt(variance_difference)
        ci_low = difference - 1.96 * standard_error
        ci_high = difference + 1.96 * standard_error
        if ci_low > 0:
            status = "Less promising"
        elif ci_high < 0:
            status = "Promising"
        else:
            status = "TBD"

    if status != "Less promising":
        promising_prices.append(price)
    comparison_rows.append([price, difference, ci_low, ci_high, status])

columns = ["price", "difference", "ci_low", "ci_high", "status"]
comparisons = pd.DataFrame(comparison_rows, columns=columns)
def highlight_tbd(row):
    if row["status"] == "TBD":
        return ["background-color: #fff2cc; color: #111111"] * len(row)
    return [""] * len(row)

comparison_display = comparisons.style.format(precision=2, na_rep="-")
comparison_display = comparison_display.apply(highlight_tbd, axis=1)
comparison_display

,price,difference,ci_low,ci_high,status
0,0,12.68,10.97,14.40,Less promising
1,5,7.68,5.97,9.40,Less promising
2,10,2.73,1.01,4.45,Less promising
3,15,4.40,2.39,6.42,Less promising
4,20,3.64,1.45,5.82,Less promising
5,25,0.00,-,-,Leader
6,30,5.14,2.64,7.65,Less promising
7,35,6.64,4.09,9.18,Less promising
8,40,7.65,5.04,10.27,Less promising
9,45,5.67,2.83,8.52,Less promising


I chose the next price points which had the highest potential or suspiciosuly close to having a value of 0 in the interval. 

In [38]:
next_prices = [12.5, 22.5, 25, 27.5, 50]
next_customer_target = customers_per_price * 2
round_2_rows = []

for price in next_prices:
    if price not in revenue_by_price:
        revenue_by_price[price] = np.array([])
    previous_revenue = revenue_by_price[price]
    customers_needed = max(0, next_customer_target - len(previous_revenue))
    if customers_needed == 0:
        continue
    purchases = sales(price, mean_customers=customers_needed, rng=rng)
    new_revenue = purchases * price
    revenue_by_price[price] = np.concatenate([previous_revenue, new_revenue])

    customers_added = len(purchases)
    revenue_added = new_revenue.sum()
    customers_at_price = len(revenue_by_price[price])
    mean_revenue = revenue_by_price[price].mean()

    total_customers += customers_added
    total_revenue += revenue_added
    round_2_rows.append([
        price, customers_needed, customers_added,
        revenue_added, customers_at_price, mean_revenue
    ])
columns = ["price", "requested_customers", "customers_added",
           "revenue_added", "customers_at_price", "mean_revenue"]
round_2_results = pd.DataFrame(round_2_rows, columns=columns)
prices = sorted(revenue_by_price)

percent_captured = 100 * total_revenue / (11.925 * total_customers)
print("Total customers:", total_customers)
print("Total revenue:", total_revenue)
print("Optimal benchmark captured (%):", round(percent_captured, 2))

round_2_results.round(2)

Total customers: 4563
Total revenue: 34562.5
Optimal benchmark captured (%): 63.52


,price,requested_customers,customers_added,revenue_added,customers_at_price,mean_revenue
0,12.5,400,401,4600.0,401,11.47
1,22.5,400,366,3847.5,366,10.51
2,25.0,195,192,2050.0,397,11.71
3,27.5,400,428,5610.0,428,13.11
4,50.0,203,206,1200.0,403,6.82


In [39]:
prices = sorted(revenue_by_price)
rows = []

for price in prices:
    customer_revenue = revenue_by_price[price]
    mean_revenue = customer_revenue.mean()
    variance = customer_revenue.var(ddof=1)
    customers = len(customer_revenue)
    rows.append([price, mean_revenue, variance, customers])

columns = ["price", "mean_revenue", "variance", "customers"]
price_summary = pd.DataFrame(rows, columns=columns)
leader_index = price_summary["mean_revenue"].idxmax()
leader_price = price_summary.loc[leader_index, "price"]
leader_revenue = revenue_by_price[leader_price]
leader_mean = leader_revenue.mean()
leader_variance = leader_revenue.var(ddof=1)
leader_customers = len(leader_revenue)
comparison_rows = []
promising_prices = []

for price in prices:
    customer_revenue = revenue_by_price[price]
    mean_revenue = customer_revenue.mean()
    customers = len(customer_revenue)
    difference = leader_mean - mean_revenue

    if price == leader_price:
        ci_low = np.nan
        ci_high = np.nan
        status = "Leader"
    else:
        variance = customer_revenue.var(ddof=1)
        variance_difference = leader_variance / leader_customers + variance / customers
        standard_error = np.sqrt(variance_difference)

        ci_low = difference - 2.576 * standard_error
        ci_high = difference + 2.576 * standard_error

        if ci_low > 0:
            status = "Less promising"
        else:
            status = "TBD"

    if status != "Less promising":
        promising_prices.append(price)
    comparison_rows.append([
        price, mean_revenue, customers, difference,
        ci_low, ci_high, status
    ])

columns = ["price", "mean_revenue", "customers", "difference",
           "ci_low", "ci_high", "status"]
comparisons = pd.DataFrame(comparison_rows, columns=columns)

print("Updated leader:", leader_price)

comparison_display = comparisons.style.format(precision=2, na_rep="-")
comparison_display = comparison_display.apply(highlight_tbd, axis=1)
comparison_display

Updated leader: 27.5


,price,mean_revenue,customers,difference,ci_low,ci_high,status
0,0.00,0.00,216,13.11,11.40,14.82,Less promising
1,5.00,5.00,176,8.11,6.40,9.82,Less promising
2,10.00,9.95,216,3.15,1.44,4.87,Less promising
3,12.50,11.47,401,1.64,-0.13,3.40,TBD
4,15.00,8.28,192,4.83,2.62,7.03,Less promising
5,20.00,9.05,210,4.06,1.59,6.53,Less promising
6,22.50,10.51,366,2.60,0.31,4.88,Less promising
7,25.00,11.71,397,1.39,-0.96,3.75,TBD
8,27.50,13.11,428,0.00,-,-,Leader
9,30.00,7.54,195,5.57,2.62,8.52,Less promising


In [40]:
customer_targets = {26.25: 400, 25: 800, 27.5: 800}
round_3_rows = []

for price, target in customer_targets.items():
    if price not in revenue_by_price:
        revenue_by_price[price] = np.array([])

    previous_revenue = revenue_by_price[price]
    customers_needed = max(0, target - len(previous_revenue))
    if customers_needed == 0:
        continue
    purchases = sales(price, mean_customers=customers_needed, rng=rng)
    new_revenue = purchases * price
    revenue_by_price[price] = np.concatenate([previous_revenue, new_revenue])
    customers_added = len(purchases)
    revenue_added = new_revenue.sum()
    customers_at_price = len(revenue_by_price[price])
    mean_revenue = revenue_by_price[price].mean()

    total_customers += customers_added
    total_revenue += revenue_added
    round_3_rows.append([
        price, customers_needed, customers_added,
        revenue_added, customers_at_price, mean_revenue
    ])

columns = ["price", "requested_customers", "customers_added",
           "revenue_added", "customers_at_price", "mean_revenue"]
round_3_results = pd.DataFrame(round_3_rows, columns=columns)
percent_captured = 100 * total_revenue / (11.925 * total_customers)

print("Total customers:", total_customers)
print("Total revenue:", total_revenue)
print("Optimal benchmark captured (%):", round(percent_captured, 2))

round_3_results.round(2)

Total customers: 5775
Total revenue: 48586.25
Optimal benchmark captured (%): 70.55


,price,requested_customers,customers_added,revenue_added,customers_at_price,mean_revenue
0,26.25,400,410,4751.25,410,11.59
1,25.00,403,402,4625.00,799,11.61
2,27.50,372,400,4647.50,828,12.39


In [41]:
prices = sorted(revenue_by_price)
rows = []

for price in prices:
    customer_revenue = revenue_by_price[price]
    mean_revenue = customer_revenue.mean()
    variance = customer_revenue.var(ddof=1)
    customers = len(customer_revenue)

    rows.append([price, mean_revenue, variance, customers])

columns = ["price", "mean_revenue", "variance", "customers"]
price_summary = pd.DataFrame(rows, columns=columns)
leader_index = price_summary["mean_revenue"].idxmax()
leader_price = price_summary.loc[leader_index, "price"]
leader_revenue = revenue_by_price[leader_price]
leader_mean = leader_revenue.mean()
leader_variance = leader_revenue.var(ddof=1)
leader_customers = len(leader_revenue)

comparison_rows = []
promising_prices = []
for price in prices:
    customer_revenue = revenue_by_price[price]
    mean_revenue = customer_revenue.mean()
    customers = len(customer_revenue)
    difference = leader_mean - mean_revenue
    if price == leader_price:
        ci_low = np.nan
        ci_high = np.nan
        status = "Leader"
    else:
        variance = customer_revenue.var(ddof=1)
        variance_difference = leader_variance / leader_customers + variance / customers
        standard_error = np.sqrt(variance_difference)

        ci_low = difference - 2.576 * standard_error
        ci_high = difference + 2.576 * standard_error
        if ci_low > 0:
            status = "Less promising"
        else:
            status = "TBD"

    if status != "Less promising":
        promising_prices.append(price)
    comparison_rows.append([
        price, mean_revenue, customers, difference,
        ci_low, ci_high, status
    ])

columns = ["price", "mean_revenue", "customers", "difference",
           "ci_low", "ci_high", "status"]
comparisons = pd.DataFrame(comparison_rows, columns=columns)
print("Updated leader:", leader_price)
comparison_display = comparisons.style.format(precision=2, na_rep="-")
comparison_display = comparison_display.apply(highlight_tbd, axis=1)
comparison_display

Updated leader: 27.5


,price,mean_revenue,customers,difference,ci_low,ci_high,status
0,0.00,0.00,216,12.39,11.16,13.61,Less promising
1,5.00,5.00,176,7.39,6.16,8.61,Less promising
2,10.00,9.95,216,2.43,1.20,3.67,Less promising
3,12.50,11.47,401,0.92,-0.39,2.22,TBD
4,15.00,8.28,192,4.11,2.25,5.96,Less promising
5,20.00,9.05,210,3.34,1.18,5.50,Less promising
6,22.50,10.51,366,1.88,-0.07,3.82,TBD
7,25.00,11.61,799,0.78,-0.89,2.45,TBD
8,26.25,11.59,410,0.80,-1.26,2.86,TBD
9,27.50,12.39,828,0.00,-,-,Leader


Now its time to turn the manual learning into a more algorithmic approach. I will define a budget of 6000 (arbitrary) customers assuming this is some kind of startup and do a inital grid search, use the leader and comapre all other possible prices using 2 sided z test then ill refine the grid to more promising targets until the optimum is reached or i run out of budget of customers!

In [42]:
auto_budget = 6000
auto_batch_size = 200
auto_rng = np.random.default_rng(101)
auto_revenue_by_price = {}
auto_batches = []

def collect_auto_batch(price, stage):
    customers_used = sum(len(values) for values in auto_revenue_by_price.values())
    remaining = auto_budget - customers_used
    if remaining <= 0:
        return
    requested = min(auto_batch_size, remaining)
    purchases = sales(price, mean_customers=requested, rng=auto_rng)
    revenue = purchases * price
    if len(revenue) == 0:
        return
    previous = auto_revenue_by_price.get(price, np.array([]))
    auto_revenue_by_price[price] = np.concatenate([previous, revenue])
    auto_batches.append([stage, price, requested, len(revenue), revenue.sum()])

def compare_auto_prices():
    rows = []
    for price in sorted(auto_revenue_by_price):
        revenue = auto_revenue_by_price[price]
        customers = len(revenue)
        mean_revenue = revenue.mean()
        variance = revenue.var(ddof=1)
        rows.append([price, mean_revenue, variance, customers])
    columns = ["price", "mean_revenue", "variance", "customers"]
    summary = pd.DataFrame(rows, columns=columns)
    summary["variance_of_mean"] = summary["variance"] / summary["customers"]
    leader_index = summary["mean_revenue"].idxmax()
    leader = summary.loc[leader_index]
    summary["difference"] = leader["mean_revenue"] - summary["mean_revenue"]
    standard_error = np.sqrt(leader["variance_of_mean"] + summary["variance_of_mean"])
    summary["ci_low"] = summary["difference"] - 2.576 * standard_error
    summary["ci_high"] = summary["difference"] + 2.576 * standard_error
    summary["status"] = "TBD"
    summary.loc[summary["ci_low"] > 0, "status"] = "Less promising"
    summary.loc[leader_index, "status"] = "Leader"
    summary.loc[leader_index, ["ci_low", "ci_high"]] = np.nan
    return summary


for price in np.arange(0, 71, 5):
    collect_auto_batch(price, "Coarse grid")

for step in [2.5, 1.25]:
    summary = compare_auto_prices()
    peak_indices = []
    for index in range(len(summary)):
        start = max(0, index - 1)
        neighbors = summary.iloc[start:index + 2]
        if summary.loc[index, "mean_revenue"] == neighbors["mean_revenue"].max():
            peak_indices.append(index)

    peaks = summary.loc[peak_indices]
    peaks = peaks.sort_values("mean_revenue", ascending=False).head(2)
    for center in peaks["price"]:
        for price in [center - step, center + step]:
            if 0 <= price <= 70 and price not in auto_revenue_by_price:
                collect_auto_batch(price, f"Refine by {step}")

while sum(len(values) for values in auto_revenue_by_price.values()) < auto_budget:
    summary = compare_auto_prices()
    contenders = summary[summary["status"] != "Less promising"]
    sample_index = contenders["variance_of_mean"].idxmax()
    price = contenders.loc[sample_index, "price"]
    collect_auto_batch(price, "Reduce uncertainty")

auto_summary = compare_auto_prices()
best_index = auto_summary["mean_revenue"].idxmax()
auto_best_price = auto_summary.loc[best_index, "price"]
auto_total_customers = sum(len(values) for values in auto_revenue_by_price.values())
auto_total_revenue = sum(values.sum() for values in auto_revenue_by_price.values())
auto_percent_captured = 100 * auto_total_revenue / (11.925 * auto_total_customers)
auto_history = pd.DataFrame(
    auto_batches,
    columns=["stage", "price", "requested_customers", "customers_added", "revenue_added"]
)

print("Automatic run selected price:", auto_best_price)
print("Automatic run customers:", auto_total_customers)
print("Automatic run revenue:", round(auto_total_revenue, 2))
print("Optimal benchmark captured (%):", round(auto_percent_captured, 2))

columns = ["price", "mean_revenue", "customers", "difference", "ci_low", "ci_high", "status"]
auto_summary[columns].round(2)

Automatic run selected price: 27.5
Automatic run customers: 6002
Automatic run revenue: 47506.25
Optimal benchmark captured (%): 66.37


,price,mean_revenue,customers,difference,ci_low,ci_high,status
0,0.00,0.00,230,13.04,11.25,14.84,Less promising
1,5.00,5.00,193,8.04,6.25,9.84,Less promising
2,10.00,9.95,217,3.09,1.29,4.89,Less promising
3,15.00,9.05,194,4.00,1.75,6.25,Less promising
4,20.00,8.42,209,4.62,2.11,7.14,Less promising
5,25.00,10.38,183,2.66,-0.30,5.62,TBD
6,26.25,12.35,389,0.70,-1.78,3.18,TBD
7,27.50,13.04,390,0.00,NaN,NaN,Leader
8,28.75,11.56,393,1.49,-1.08,4.05,TBD
9,30.00,10.48,395,2.56,-0.02,5.14,TBD
